# 5 — Scaling past two dimensions

Bivariate copulas are where the intuition lives, but real books have more than two
instruments. This notebook covers what changes when $d$ grows: what stays easy, what gets
expensive, and where a flow starts to lose to a vine.

The headline, established by the benchmark in this repository, is worth stating up front:

> **Dimension alone does not decide the winner — family match does.** In 5-D, a vine beats
> the flow comfortably on elliptical data and loses to it comfortably on a mixture. What
> changes with $d$ is the *sample size you need*, not the ranking.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from neurocopula import NeuroCopula, datasets, metrics, plotting as ncplot, theme

pd.set_option("display.width", 150, "display.precision", 4)

# Apply the library's visual theme, so hand-rolled figures in this notebook
# match the ones the library produces. rcParams are read when an Axes is
# created, so this must run before any plotting.
theme.apply_theme()

## The panel

Five instruments in two blocks, plus a near-independent FX series. Generated from a
Student-t copula with a block correlation matrix, so it has genuine tail dependence and a
known structure.

In [ ]:
df = datasets.make_asset_panel(n=4000, seed=0)
print(df.shape)
print()
print("Kendall's tau of the data:")
metrics.kendall_tau_matrix(df).round(3)

The block structure is visible: within-block τ ≈ 0.5, cross-block τ ≈ 0.05, with `fx_A` weakly
attached to the metals only.

## Sampling mode matters now

In `"autoregressive"` mode the flow samples one dimension at a time — the most expressive
option, but the cost grows with $d$ and every query method draws large samples internally. In
`"coupling"` mode all dimensions are sampled in one pass.

Let us measure rather than assert.

In [ ]:
import time

timings = []
for mode in ["autoregressive", "coupling"]:
    t0 = time.perf_counter()
    m = NeuroCopula(transforms=4, hidden_features=(64, 64), copula_mode=True,
                    sampling_mode=mode, seed=0).fit(
        df, epochs=400, patience=50, split_method="random", verbose=False)
    fit_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    m.sample(200_000)
    sample_s = time.perf_counter() - t0

    timings.append({"sampling_mode": mode, "fit seconds": fit_s,
                    "200k draws (seconds)": sample_s,
                    "held-out copula loglik": m.score(df, copula=True)})
pd.DataFrame(timings).set_index("sampling_mode").round(3)

Coupling is markedly faster to sample from, at little or no cost in fit quality on this data.
Since `tail_dependence`, `conditional_probability` and `joint_exceedance_probability` all draw
large samples internally, that difference compounds across a session. **Use `"coupling"`
whenever $d > 2$ or you plan many queries.**

In [ ]:
model = NeuroCopula(
    transforms=4, hidden_features=(64, 64),
    copula_mode=True, sampling_mode="coupling", seed=0,
).fit(df, epochs=700, patience=60, split_method="random", verbose=True)

## Did it get the structure right?

With five columns there are ten pairs. Checking each by eye does not scale; the tau error
matrix does it at a glance — a pale panel means the dependence is right.

In [ ]:
samples = model.sample(40_000)
fig = ncplot.plot_tau_error_heatmap(df, {"NeuroCopula": samples})
plt.show()

err = (metrics.kendall_tau_matrix(samples) - metrics.kendall_tau_matrix(df))
offdiag = ~np.eye(len(df.columns), dtype=bool)
print(f"mean absolute tau error across all 10 pairs : {np.abs(err.values[offdiag]).mean():.4f}")
print(f"worst pair                                  : {np.abs(err.values[offdiag]).max():.4f}")

In [ ]:
fig = ncplot.plot_pairwise_grid({"data": df, "NeuroCopula": samples}, n=1500)
plt.show()

The full pairwise view. Look for pairs where the orange (model) cloud has a different *shape*
from the blue (data) cloud, not just a different spread — shape differences are dependence
errors, spread differences would be marginal errors, which copula mode rules out by
construction.

## Tail dependence, pair by pair

The summary that matters for risk. We compute λ_L for every pair, in the data and in the
model.

In [ ]:
cols = list(df.columns)
u_data = model.pseudo_observations(df)
u_model = model.sample_uniform(200_000)

rows = []
for i, a in enumerate(cols):
    for b in cols[i + 1:]:
        rows.append({
            "pair": f"{a} / {b}",
            "tau": metrics.kendall_tau_matrix(df).loc[a, b],
            "data lambda_L": metrics.tail_dependence(u_data[a], u_data[b], q=0.02),
            "model lambda_L": metrics.tail_dependence(u_model[a], u_model[b], q=0.02),
        })
pairs = pd.DataFrame(rows).set_index("pair")
pairs["abs error"] = (pairs["model lambda_L"] - pairs["data lambda_L"]).abs()
pairs.sort_values("tau", ascending=False).round(3)

Two things to notice. First, tail dependence does not follow τ mechanically — a pair can have
modest rank correlation and still meaningful joint-tail behaviour. Second, the model's errors
are largest on the weakly-dependent pairs, where the estimate itself is noisiest (only ~2% of
draws land in the tail, and the pairs are nearly independent there).

## The scaling question

How does fit quality change as $d$ grows, holding the number of rows fixed? This is the
question that actually decides whether a flow is appropriate for your problem.

In [ ]:
from neurocopula import VineCopula
from neurocopula.benchmark import _split

rows = []
for d in [2, 3, 5, 8]:
    data = datasets.make_t_copula(n=3000, rho=0.6, dof=4, d=d, seed=0)
    train, test = _split(data, 0.25, seed=0)

    flow = NeuroCopula(transforms=4, hidden_features=(64, 64), copula_mode=True,
                       marginal_density="none", sampling_mode="coupling", seed=0).fit(
        train, epochs=500, patience=50, split_method="random", verbose=False)
    vine = VineCopula(family_set="parametric", marginal_density="none").fit(train)

    rows.append({
        "d": d,
        "flow params": sum(p.numel() for p in flow.flow.parameters()),
        "vine params": vine.vine.npars,
        "flow loglik": flow.score(test, copula=True),
        "vine loglik": vine.score(test, copula=True),
    })
    print(f"d={d}: flow {rows[-1]['flow loglik']:.4f}   vine {rows[-1]['vine loglik']:.4f}")

scaling = pd.DataFrame(rows).set_index("d")
scaling["flow - vine"] = scaling["flow loglik"] - scaling["vine loglik"]
scaling.round(4)

This is elliptical data, so the vine has exactly the right family — its home turf. Watch the
`flow - vine` column get worse as $d$ grows: the flow's parameter count climbs steeply while
the row count stays fixed, so it becomes progressively more sample-starved against a model
that needs only a handful of parameters.

**This is a real limitation, and the honest conclusion is: on elliptical data in higher
dimensions with limited rows, use a vine.**

Now the same experiment where no parametric family fits.

In [ ]:
rows = []
for d in [2, 3, 5, 8]:
    data = datasets.make_mixed_regime(n=3000, d=d, seed=0)
    train, test = _split(data, 0.25, seed=0)

    flow = NeuroCopula(transforms=4, hidden_features=(64, 64), copula_mode=True,
                       marginal_density="none", sampling_mode="coupling", seed=0).fit(
        train, epochs=500, patience=50, split_method="random", verbose=False)
    vine = VineCopula(family_set="parametric", marginal_density="none").fit(train)

    rows.append({"d": d, "flow loglik": flow.score(test, copula=True),
                 "vine loglik": vine.score(test, copula=True)})
    print(f"d={d}: flow {rows[-1]['flow loglik']:.4f}   vine {rows[-1]['vine loglik']:.4f}")

mixed = pd.DataFrame(rows).set_index("d")
mixed["flow - vine"] = mixed["flow loglik"] - mixed["vine loglik"]
mixed.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.axhline(0, color="#52514e", lw=1.2)
ax.plot(scaling.index, scaling["flow - vine"], marker="o", color="#2a78d6",
        label="Student-t copula (a family the vine has)")
ax.plot(mixed.index, mixed["flow - vine"], marker="o", color="#eb6834",
        label="two-regime mixture (no matching family)")
ax.set_xlabel("dimension d"); ax.set_ylabel("flow loglik - vine loglik")
ax.set_title("Which model wins depends on family match, not dimension", loc="left",
             fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

Two lines with opposite signs, at every dimension tested. The blue line (elliptical data) sits
below zero and drifts lower; the orange line (mixture) sits above zero throughout. Dimension
shifts the magnitude; **family match decides the sign**.

## Practical guidance

| Situation | Recommendation |
|---|---|
| $d \le 3$, any structure | Either works; the flow is safe and needs no family choice |
| $d$ large, plausibly elliptical | **Vine** — far more sample-efficient |
| $d$ large, asymmetric or regime-switching | **Flow** — no family can express it |
| Many rows relative to $d$ | Flow's disadvantage fades |
| Few rows relative to $d$ | Vine, or reduce the flow's capacity |
| Unsure | Fit both — they share an API — and compare held-out `score(..., copula=True)` |

For a flow in higher dimensions: use `sampling_mode="coupling"`, consider fewer `transforms`
and narrower `hidden_features` to cut parameters, and add `weight_decay` (1e-5 to 1e-4).

## Takeaways

1. **`sampling_mode="coupling"` is the default choice past two dimensions.**
2. **Use the tau error heatmap** to check ten pairs at once instead of by eye.
3. **Tail dependence does not follow τ** — check it per pair.
4. **Higher $d$ costs the flow sample efficiency, not correctness.**
5. **Family match, not dimension, decides which model wins.** Fit both; the API is shared.